# Pooling Bootstrap Tests

This notebook runs a focused paired bootstrap robustness test for the main thesis configuration: contextual BERTurk, layer 7. It uses the existing pair-similarity CSV only and compares first-subtoken pooling against mean, last, and max pooling.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

PROJECT_ROOT = Path.cwd()
INPUT_PATH = PROJECT_ROOT / "outputs" / "results" / "0501-contextual_all_models_pair_similarities.csv"
OUTPUT_PATH = PROJECT_ROOT / "outputs" / "tables" / "1001-pooling_bootstrap_tests.csv"

INPUT_CONDITION = "contextual"
MODEL = "BERTurk"
LAYER = 7
POOLINGS = ["first", "mean", "last", "max"]
COMPARISON_POOLINGS = ["mean", "last", "max"]

SEED = 20260522
N_BOOTSTRAP = 5000

pd.set_option("display.float_format", "{:.6f}".format)

## Load and Inspect Columns

In [ ]:
pair_similarities = pd.read_csv(INPUT_PATH)

print(f"Loaded {INPUT_PATH}")
print(f"Shape: {pair_similarities.shape}")
print("Columns:")
for column in pair_similarities.columns:
    print(f"- {column}")

pair_similarities.head()

## Filter Main Configuration and Pivot Pooling Scores

In [ ]:
required_columns = {
    "QID",
    "W1",
    "W2",
    "Sim",
    "input_condition",
    "model",
    "layer",
    "pooling",
    "cosine_similarity",
}
missing_columns = required_columns - set(pair_similarities.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

main_config = pair_similarities[
    (pair_similarities["input_condition"] == INPUT_CONDITION)
    & (pair_similarities["model"] == MODEL)
    & (pair_similarities["layer"] == LAYER)
].copy()

if main_config.empty:
    raise ValueError("No rows found for the requested main configuration.")

duplicate_mask = main_config.duplicated(subset=["QID", "W1", "W2", "Sim", "pooling"], keep=False)
if duplicate_mask.any():
    raise ValueError("Duplicate pair/pooling rows found; inspect before pivoting.")

pooling_wide = (
    main_config
    .pivot(index=["QID", "W1", "W2", "Sim"], columns="pooling", values="cosine_similarity")
    .reset_index()
)
pooling_wide.columns.name = None

missing_poolings = [pooling for pooling in POOLINGS if pooling not in pooling_wide.columns]
if missing_poolings:
    raise ValueError(f"Missing pooling columns after pivot: {missing_poolings}")

pooling_wide = (
    pooling_wide[["QID", "W1", "W2", "Sim"] + POOLINGS]
    .dropna(subset=["Sim"] + POOLINGS)
    .sort_values("QID")
    .reset_index(drop=True)
)

print(f"Filtered rows: {len(main_config)}")
print(f"Pivoted word pairs: {len(pooling_wide)}")
print(pooling_wide[POOLINGS].isna().sum())

pooling_wide.head()

## Paired Bootstrap Test

In [ ]:
def spearman_rho(gold_scores: np.ndarray, model_scores: np.ndarray) -> float:
    """Return Spearman rho as a plain float."""
    return float(spearmanr(gold_scores, model_scores).statistic)


def bootstrap_pooling_comparison(
    frame: pd.DataFrame,
    other_pooling: str,
    n_bootstrap: int = N_BOOTSTRAP,
    seed: int = SEED,
) -> dict:
    """Compare first pooling with another pooling strategy over paired bootstrap resamples."""
    gold = frame["Sim"].to_numpy()
    first_scores = frame["first"].to_numpy()
    other_scores = frame[other_pooling].to_numpy()
    n_pairs = len(frame)

    rho_first = spearman_rho(gold, first_scores)
    rho_other = spearman_rho(gold, other_scores)
    observed_delta = rho_first - rho_other

    rng = np.random.default_rng(seed)
    bootstrap_deltas = np.empty(n_bootstrap, dtype=float)

    for bootstrap_index in range(n_bootstrap):
        sampled_indices = rng.integers(0, n_pairs, size=n_pairs)
        rho_first_bootstrap = spearman_rho(gold[sampled_indices], first_scores[sampled_indices])
        rho_other_bootstrap = spearman_rho(gold[sampled_indices], other_scores[sampled_indices])
        bootstrap_deltas[bootstrap_index] = rho_first_bootstrap - rho_other_bootstrap

    ci_95_low, ci_95_high = np.percentile(bootstrap_deltas, [2.5, 97.5])
    count_delta_leq_0 = int(np.sum(bootstrap_deltas <= 0))
    count_delta_geq_0 = int(np.sum(bootstrap_deltas >= 0))
    p_lower_or_equal_zero = count_delta_leq_0 / n_bootstrap
    p_greater_or_equal_zero = count_delta_geq_0 / n_bootstrap
    p_two_sided = min(2 * min(p_lower_or_equal_zero, p_greater_or_equal_zero), 1.0)
    p_one_sided_corrected = (count_delta_leq_0 + 1) / (n_bootstrap + 1)
    p_greater_or_equal_zero_corrected = (count_delta_geq_0 + 1) / (n_bootstrap + 1)
    p_two_sided_corrected = min(
        1.0,
        2 * min(p_one_sided_corrected, p_greater_or_equal_zero_corrected),
    )
    p_one_sided_report = (
        f"p < {1 / n_bootstrap:.6f}" if count_delta_leq_0 == 0 else f"{p_lower_or_equal_zero:.6f}"
    )
    p_two_sided_report = (
        f"p < {2 / n_bootstrap:.6f}" if min(count_delta_leq_0, count_delta_geq_0) == 0 else f"{p_two_sided:.6f}"
    )

    return {
        "model": MODEL,
        "layer": LAYER,
        "comparison": f"first_vs_{other_pooling}",
        "rho_first": rho_first,
        "rho_other": rho_other,
        "delta_rho_observed": observed_delta,
        "ci_95_low": float(ci_95_low),
        "ci_95_high": float(ci_95_high),
        "p_one_sided": p_one_sided_report,
        "p_two_sided": p_two_sided_report,
        "p_one_sided_corrected": float(p_one_sided_corrected),
        "p_two_sided_corrected": float(p_two_sided_corrected),
        "significant_ci_excludes_zero": bool((ci_95_low > 0) or (ci_95_high < 0)),
        "significant_p_two_sided": bool(p_two_sided_corrected < 0.05),
        "n_pairs": n_pairs,
        "n_bootstrap": n_bootstrap,
    }

## Run Comparisons and Save Output

In [ ]:
results = pd.DataFrame(
    [bootstrap_pooling_comparison(pooling_wide, pooling) for pooling in COMPARISON_POOLINGS]
)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
results.to_csv(OUTPUT_PATH, index=False)

print(f"Saved bootstrap test table to {OUTPUT_PATH}")
results

## Interpretation

In [ ]:
for row in results.itertuples(index=False):
    direction = "higher" if row.delta_rho_observed > 0 else "lower"
    ci_statement = "excludes zero" if row.significant_ci_excludes_zero else "includes zero"
    p_statement = "significant" if row.significant_p_two_sided else "not significant"
    print(
        f"{row.comparison}: first pooling is {direction} by delta_rho="
        f"{row.delta_rho_observed:.4f}. The 95% bootstrap CI "
        f"[{row.ci_95_low:.4f}, {row.ci_95_high:.4f}] {ci_statement}; "
        f"empirical one-sided {row.p_one_sided}, empirical two-sided {row.p_two_sided}; "
        f"plus-one corrected one-sided p={row.p_one_sided_corrected:.6f}, "
        f"plus-one corrected two-sided p={row.p_two_sided_corrected:.6f} "
        f"({p_statement})."
    )